# AWS Bedrock Starter Notebook for Transaction Categorisation POC

This notebook demonstrates how to use Claude models via AWS Bedrock for transaction categorisation.

## 1. Setup & Installation

In [ ]:
# Install required packages (uncomment if needed)
# !pip install boto3 python-dotenv --upgrade

## 2. Load API Key from .env

Create a `.env` file in your notebook directory with:
```
AWS_BEARER_TOKEN_BEDROCK=your-api-key-here
```

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv(override=True)

# Verify API key is loaded
api_key = os.environ.get('AWS_BEARER_TOKEN_BEDROCK')
if api_key:
    print(f"✓ API key loaded: {api_key[:8]}...{api_key[-4:]}")
else:
    raise ValueError("✗ API key not found - check your .env file")

## 3. Configuration

Using API key authentication. Short-term keys expire after ~12 hours.

In [ ]:
import boto3
import json
from typing import Optional

AWS_REGION = "ap-southeast-2"  # Sydney

# Models available in Sydney
MODELS = {
    "claude_35_sonnet_v2": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    "claude_3_haiku": "anthropic.claude-3-haiku-20240307-v1:0",
    "claude_3_sonnet": "anthropic.claude-3-sonnet-20240229-v1:0",
}

DEFAULT_MODEL = MODELS["claude_35_sonnet_v2"]

## 4. Bedrock Client

In [ ]:
class BedrockClient:
    """Simple wrapper for Bedrock inference."""
    
    def __init__(self, region: str = AWS_REGION, model_id: str = DEFAULT_MODEL):
        self.client = boto3.client("bedrock-runtime", region_name=region)
        self.model_id = model_id
        
    def invoke(self, prompt: str, system: Optional[str] = None,
               max_tokens: int = 1024, temperature: float = 0.0) -> dict:
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]
        }
        if system:
            body["system"] = system
        
        response = self.client.invoke_model(
            modelId=self.model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps(body)
        )
        
        result = json.loads(response["body"].read())
        return {
            "content": result["content"][0]["text"],
            "usage": result.get("usage", {}),
            "model": self.model_id,
            "stop_reason": result.get("stop_reason")
        }

# 4A. Read input dataset

In [ ]:
!ls ../release_testing/v7_1/testset_v2_enrich_circleci_shared_v7.csv

In [ ]:
df = pd.read_csv("../release_testing/v7_1/testset_v2_enrich_circleci_shared_v7.csv")

In [ ]:
df.columns

In [ ]:
sel_col = [#'id', 'provider_external_id', 
'provider_name', 'aggregator', 'account_type', 'base_type','amount', 'original_description','cdr_biller_code'
]
#, 'merchant_name', 'base_category_key', 'has_merch_ideasv2', 'category_ideasv2']

In [ ]:
df.shape

In [ ]:
df.category_ideasv2.unique()

In [ ]:
df.base_category_key.value_counts()

In [ ]:
pd.set_option('display.max_colwidth', 100)

In [ ]:
df[sel_col].head(20)

## 5. Test Connection

In [ ]:
bedrock = BedrockClient()

response = bedrock.invoke("Say 'Bedrock connection successful!' and nothing else.")
print(f"Response: {response['content']}")
print(f"Tokens used: {response['usage']}")

## 6. Transaction Categorisation Setup

In [ ]:
CATEGORY_TAXONOMY = """
Categories:
1. GROCERIES - Supermarkets, food stores, grocery delivery
2. DINING - Restaurants, cafes, fast food, food delivery apps
3. TRANSPORT - Fuel, public transport, rideshare, parking
4. UTILITIES - Electricity, gas, water, internet, phone
5. ENTERTAINMENT - Streaming, movies, games, concerts
6. SHOPPING - Retail, clothing, electronics, online marketplaces
7. HEALTH - Pharmacy, medical, fitness, wellness
8. TRAVEL - Hotels, flights, car rental, travel agencies
9. FINANCIAL - Bank fees, insurance, investments
10. OTHER - Uncategorised transactions
"""

BUSINESS_RULES = """
Rules:
- "UBER" or "LYFT" → TRANSPORT (not DINING even if Uber Eats)
- Amounts < $20 at restaurants → likely DINING
- "PRIME" or "AMZN" → SHOPPING unless clearly a streaming charge
- Recurring monthly charges → likely UTILITIES or ENTERTAINMENT
"""

FEW_SHOT_EXAMPLES = """
Examples:
Transaction: "WOOLWORTHS 1234 SYDNEY" $85.50 → GROCERIES
Transaction: "UBER *TRIP HELP.UBER.COM" $23.40 → TRANSPORT
Transaction: "NETFLIX.COM" $22.99 → ENTERTAINMENT
Transaction: "BP SERVICE STATION" $65.00 → TRANSPORT
Transaction: "MENULOG*THAI PALACE" $42.00 → DINING
"""

In [ ]:
def build_categorisation_prompt(transaction_desc: str, amount: float,
                                  taxonomy: str = CATEGORY_TAXONOMY,
                                  rules: str = BUSINESS_RULES,
                                  examples: str = FEW_SHOT_EXAMPLES) -> tuple:
    system_prompt = f"""You are a financial transaction categorisation system.
Your task is to categorise bank transactions into the correct category.

{taxonomy}

{rules}

{examples}

Respond with ONLY the category name. Be consistent and precise."""

    user_prompt = f'Categorise this transaction: "{transaction_desc}" Amount: ${amount:.2f}'
    return system_prompt, user_prompt


def categorise_transaction(description: str, amount: float,
                           client: BedrockClient = None, **kwargs) -> dict:
    if client is None:
        client = BedrockClient()
    
    system, user = build_categorisation_prompt(description, amount, **kwargs)
    response = client.invoke(prompt=user, system=system, max_tokens=100, temperature=0.0)
    
    return {
        "transaction": description,
        "amount": amount,
        "category": response["content"].strip(),
        "tokens": response["usage"],
        "model": response["model"]
    }

## 7. Test Categorisation

In [ ]:
test_transactions = [
    ("COLES SUPERMARKET MELBOURNE", 125.50),
    ("UBER *TRIP HELP.UBER.COM", 18.90),
    ("SPOTIFY SYDNEY", 12.99),
    ("SHELL SERVICE STATION", 78.00),
    ("AMAZON PRIME*1234", 9.99),
    ("DR SMITH MEDICAL CTR", 85.00),
]

print("Transaction Categorisation Results")
print("=" * 50)

total_tokens = {"input_tokens": 0, "output_tokens": 0}

for desc, amount in test_transactions:
    result = categorise_transaction(desc, amount, client=bedrock)
    print(f"\n{desc} (${amount})")
    print(f"  → {result['category']}")
    print(f"  Tokens: {result['tokens']}")
    
    total_tokens["input_tokens"] += result["tokens"].get("input_tokens", 0)
    total_tokens["output_tokens"] += result["tokens"].get("output_tokens", 0)

print(f"\n{'=' * 50}")
print(f"Total tokens used: {total_tokens}")

## 8. Cost Estimation

In [ ]:
PRICING = {
    "anthropic.claude-3-5-sonnet-20241022-v2:0": {"input": 0.003, "output": 0.015},
    "anthropic.claude-3-haiku-20240307-v1:0": {"input": 0.00025, "output": 0.00125},
    "anthropic.claude-3-sonnet-20240229-v1:0": {"input": 0.003, "output": 0.015},
}

def estimate_cost(input_tokens: int, output_tokens: int, model_id: str) -> dict:
    prices = PRICING.get(model_id, PRICING["anthropic.claude-3-5-sonnet-20241022-v2:0"])
    input_cost = (input_tokens / 1000) * prices["input"]
    output_cost = (output_tokens / 1000) * prices["output"]
    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost,
        "cost_per_1k_transactions": (input_cost + output_cost) * 1000 / len(test_transactions)
    }

cost = estimate_cost(total_tokens["input_tokens"], total_tokens["output_tokens"], DEFAULT_MODEL)
print(f"Cost for {len(test_transactions)} transactions: ${cost['total_cost']:.6f}")
print(f"Projected cost per 1,000 transactions: ${cost['cost_per_1k_transactions']:.4f}")

## 9. Next Steps

1. **Experiment with taxonomy** - modify `CATEGORY_TAXONOMY`
2. **Test different prompts** - create variations of the prompt templates
3. **Load production data** - anonymised transactions with known labels
4. **Run batch comparison** - measure accuracy vs production labels
5. **Compare models** - test Haiku vs Sonnet for cost/accuracy tradeoff